# задание 3

In [ ]:
format elf64
public _start

include '../MyLibrary.asm'

section '.data' writeable

  nl db 10       ;символ перевода строки
  need_nl db 0   ;1 - если файл не заканчивается переводом строки

section '.bss' writable

  buf rb 100 ;буфер для чтения кусков файла

section '.text' executable

_start:
  ;Считываем количество пааметров командной строки
   pop rcx
   cmp rcx, 3
   jl .l1

   ;открываем файл, который будем читать
   mov rdi,[rsp+8]
   mov rax, 2
   mov rsi, 0o
   syscall
   cmp rax, 0
   jl .l1
   ;Сохраняем дескриптор
   mov r8, rax

   ;открываем файл, в который будем записывать
   mov rdi,[rsp+16]
   mov rax, 2
   mov rsi, 577
   mov rdx, 777o
   syscall
   cmp rax, 0
   jl .l1
   ;Сохраняем дескриптор
   mov r9, rax

   ;Узнаём размер файла: lseek(fd, 0, SEEK_END)
   mov rax, 8
   mov rdi, r8
   mov rsi, 0
   mov rdx, 2
   syscall
   cmp rax, 0
   jle .close ;пустой файл - записывать нечего

   ;r12 - конец текущей строки (смещение за её последним символом)
   ;r13 - смещение, левее которого ищем '\n'
   mov r12, rax
   mov r13, rax

   ;Читаем последний символ файла
   mov rsi, rax
   dec rsi
   mov rax, 8
   mov rdi, r8
   mov rdx, 0
   syscall
   mov rax, 0
   mov rdi, r8
   mov rsi, buf
   mov rdx, 1
   syscall

   cmp byte [buf], 10
   jne @f
   dec r13 ;последний '\n' относится к последней строке, пропускаем его
   jmp .search
   @@:
   mov byte [need_nl], 1

   ;Идём от конца файла к началу кусками по 100 байт
   .search:
     cmp r13, 0
     je .write_first

     ;r14 - начало куска: max(r13 - 100, 0)
     mov r14, r13
     sub r14, 100
     cmp r14, 0
     jge @f
     mov r14, 0
     @@:

     ;Читаем кусок [r14, r13) в buf
     mov rax, 8
     mov rdi, r8
     mov rsi, r14
     mov rdx, 0
     syscall
     mov rax, 0
     mov rdi, r8
     mov rsi, buf
     mov rdx, r13
     sub rdx, r14
     syscall

     ;rbx - индекс в buf, ищем '\n' с конца куска
     mov rbx, r13
     sub rbx, r14
     .scan:
       cmp rbx, 0
       je .next_chunk
       dec rbx
       cmp byte [buf + rbx], 10
       jne .scan

     ;Нашли '\n' на смещении r14+rbx - строка начинается сразу за ним
     mov r13, r14
     add r13, rbx
     mov rsi, r13
     inc rsi
     mov rdi, r12
     call copy_part
     ;конец следующей строки - сразу за найденным '\n'
     mov r12, r13
     inc r12
     jmp .search

     .next_chunk:
       mov r13, r14
       jmp .search

   ;первая строка файла: с 0 по r12 (не включая)
   .write_first:
     mov rsi, 0
     mov rdi, r12
     call copy_part

   .close:
   mov rdi, r8
   mov rax, 3
   syscall

   mov rdi, r9
   mov rax, 3
   syscall

.l1:
   call exit

;Копирует часть входного файла (r8) в выходной (r9) кусками по 100 байт
;input: rsi - смещение начала, rdi - смещение конца (не включая)
copy_part:
  push rbx
  push r15

  mov rbx, rsi ;текущее смещение
  mov r15, rdi ;смещение конца

  ;lseek(fd, начало, SEEK_SET)
  mov rax, 8
  mov rdi, r8
  mov rdx, 0
  syscall

  .loop:
    cmp rbx, r15
    jge .end

    ;читаем min(100, конец - текущее) байт
    mov rdx, r15
    sub rdx, rbx
    cmp rdx, 100
    jle @f
    mov rdx, 100
    @@:
    mov rax, 0
    mov rdi, r8
    mov rsi, buf
    syscall
    cmp rax, 0
    jle .end
    add rbx, rax

    mov rdx, rax
    mov rax, 1
    mov rdi, r9
    mov rsi, buf
    syscall
    jmp .loop
  .end:

  ;Первой записывается последняя строка файла - если у неё не было '\n', дописываем
  cmp byte [need_nl], 1
  jne @f
  mov rax, 1
  mov rdi, r9
  mov rsi, nl
  mov rdx, 1
  syscall
  mov byte [need_nl], 0
  @@:

  pop r15
  pop rbx
  ret


The installation, called Bring Kids Back, made its Sacramento debut as part of an ongoing campaign to draw attention to the thousands of children and support international efforts for their return. It was displayed on the National Mall in Washington, D.C., and in Los Angeles earlier this year.
I think everybody can agree that stealing children is an atrocity and should not be happening in this day and age, said Aleksandr Krapivkin, director of advocacy and engagement at Nova Ukraine and one of the event moderators.
If we dont want this to ever happen again not only in Ukraine,
but  to any nation, then we have to take action now.

but  to any nation, then we have to take action now.
If we dont want this to ever happen again not only in Ukraine,
I think everybody can agree that stealing children is an atrocity and should not be happening in this day and age, said Aleksandr Krapivkin, director of advocacy and engagement at Nova Ukraine and one of the event moderators.
The installation, called Bring Kids Back, made its Sacramento debut as part of an ongoing campaign to draw attention to the thousands of children and support international efforts for their return. It was displayed on the National Mall in Washington, D.C., and in Los Angeles earlier this year.


# задание 10

In [1]:
; алгоритм:
;   1. Открываем входной файл на чтение (r8) и выходной на запись (r9).
;   2. Читаем входной файл вперёд кусками по 100 байт в buf. r15 - смещение
;      текущего куска в файле, перед каждым чтением ставим позицию через
;      lseek, потому что функции копирования её сдвигают.
;   3. Просматриваем кусок по символам:
;        - буква: запоминаем r13 (если это первая буква) и r14;
;        - пробельный символ: ничего не делаем;
;        - '.', '!', '?': предложение закончилось - вызываем write_sentence,
;          которая записывает его в выходной файл и начинает новое.
;   4. В конце файла записываем остаток (предложение без знака конца).
format elf64
public _start

include '../MyLibrary.asm'

section '.bss' writable

  buf rb 100  ;текущий кусок входного файла
  cbuf rb 100 ;буфер для функций копирования

section '.text' executable

_start:
   pop rcx
   cmp rcx, 3
   jl .l1

   ;открываем входной файл на чтение
   mov rdi,[rsp+8]
   mov rax, 2
   mov rsi, 0o
   syscall
   cmp rax, 0
   jl .l1
   mov r8, rax

   ;открываем выходной файл: O_WRONLY | O_CREAT | O_TRUNC
   mov rdi,[rsp+16]
   mov rax, 2
   mov rsi, 577
   mov rdx, 777o
   syscall
   cmp rax, 0
   jl .l1
   mov r9, rax

   xor r12, r12 ;начало предложения
   mov r13, -1  ;первая буква (ещё не было)
   xor r14, r14 ;за последней буквой
   xor r15, r15 ;смещение текущего куска

   .read_chunk:
     ;lseek(r8, r15, SEEK_SET) и читаем до 100 байт
     mov rax, 8
     mov rdi, r8
     mov rsi, r15
     mov rdx, 0
     syscall
     mov rax, 0
     mov rdi, r8
     mov rsi, buf
     mov rdx, 100
     syscall
     cmp rax, 0
     jle .eof
     mov r10, rax ;длина куска

     xor rbx, rbx
     .scan:
       cmp rbx, r10
       jge .chunk_done

       mov al, [buf + rbx]
       cmp al, '.'
       je .term
       cmp al, '!'
       je .term
       cmp al, '?'
       je .term
       cmp al, ' '
       jle .next ;пробельный символ

       ;буква: rcx - её смещение в файле
       mov rcx, r15
       add rcx, rbx
       cmp r13, 0
       jge @f
       mov r13, rcx
       @@:
       inc rcx
       mov r14, rcx
       jmp .next

       .term:
         ;конец предложения - сразу за знаком
         mov rcx, r15
         add rcx, rbx
         inc rcx
         call write_sentence

       .next:
         inc rbx
         jmp .scan

     .chunk_done:
       add r15, r10
       jmp .read_chunk

   ;конец файла: записываем остаток, r15 - размер файла
   .eof:
     mov rcx, r15
     call write_sentence

   mov rdi, r8
   mov rax, 3
   syscall

   mov rdi, r9
   mov rax, 3
   syscall

.l1:
   call exit

;Записывает предложение в выходной файл и начинает новое
;input: r12 - начало, r13 - первая буква (-1 если нет), r14 - за последней буквой,
;       rcx - конец предложения (не включая)
;output: r12 = rcx, r13 = -1
write_sentence:
  push rbx
  mov rbx, rcx ;syscall портит rcx

  cmp r13, 0
  jge @f
  ;букв нет - копируем как есть
  mov rsi, r12
  mov rdi, rbx
  call copy_forward
  jmp .done

  @@:
  mov rsi, r12
  mov rdi, r13
  call copy_forward
  mov rsi, r13
  mov rdi, r14
  call copy_reversed
  mov rsi, r14
  mov rdi, rbx
  call copy_forward

  .done:
  mov r12, rbx
  mov r13, -1
  pop rbx
  ret

;Копирует часть входного файла (r8) в выходной (r9) как есть
;input: rsi - смещение начала, rdi - смещение конца (не включая)
copy_forward:
  push rbx
  push r15

  mov rbx, rsi
  mov r15, rdi

  mov rax, 8
  mov rdi, r8
  mov rdx, 0
  syscall

  .loop:
    cmp rbx, r15
    jge .end

    ;читаем min(100, осталось) байт
    mov rdx, r15
    sub rdx, rbx
    cmp rdx, 100
    jle @f
    mov rdx, 100
    @@:
    mov rax, 0
    mov rdi, r8
    mov rsi, cbuf
    syscall
    cmp rax, 0
    jle .end
    add rbx, rax

    mov rdx, rax
    mov rax, 1
    mov rdi, r9
    mov rsi, cbuf
    syscall
    jmp .loop
  .end:

  pop r15
  pop rbx
  ret

;Копирует часть входного файла (r8) в выходной (r9) задом наперёд
;input: rsi - смещение начала, rdi - смещение конца (не включая)
copy_reversed:
  push rbx
  push r14
  push r15

  mov rbx, rsi ;начало
  mov r15, rdi ;конец ещё не записанной части

  .loop:
    cmp r15, rbx
    jle .end

    ;кусок [r14, r15), r14 = max(r15 - 100, начало)
    mov r14, r15
    sub r14, 100
    cmp r14, rbx
    jge @f
    mov r14, rbx
    @@:

    mov rax, 8
    mov rdi, r8
    mov rsi, r14
    mov rdx, 0
    syscall
    mov rax, 0
    mov rdi, r8
    mov rsi, cbuf
    mov rdx, r15
    sub rdx, r14
    syscall
    cmp rax, 0
    jle .end

    ;разворачиваем кусок и записываем
    mov rdx, rax
    mov rsi, cbuf
    mov rdi, cbuf
    add rdi, rax
    dec rdi
    call reverse_str

    mov rax, 1
    mov rdi, r9
    mov rsi, cbuf
    syscall

    mov r15, r14
    jmp .loop
  .end:

  pop r15
  pop r14
  pop rbx
  ret


SyntaxError: invalid octal literal (4167486141.py, line 21)

The installation, called Bring Kids Back, made its Sacramento debut as part of an ongoing campaign to draw attention to the thousands of children and support international efforts for their return. It was displayed on the National Mall in Washington, D.C., and in Los Angeles earlier this year.
I think everybody can agree that stealing children is an atrocity and should not be happening in this day and age, said Aleksandr Krapivkin, director of advocacy and engagement at Nova Ukraine and one of the event moderators.
If we dont want this to ever happen again not only in Ukraine,
but  to any nation, then we have to take action now.

nruter rieht rof stroffe lanoitanretni troppus dna nerdlihc fo sdnasuoht eht ot noitnetta ward ot ngiapmac gniogno na fo trap sa tubed otnemarcaS sti edam ,kcaB sdiK gnirB dellac ,noitallatsni ehT. D ,notgnihsaW ni llaM lanoitaN eht no deyalpsid saw tI.C.raey siht reilrae selegnA soL ni dna ,.
srotaredom tneve eht fo eno dna eniarkU avoN ta tnemegagne dna ycacovda fo rotcerid ,nikviparK rdnaskelA dias ,ega dna yad siht ni gnineppah eb ton dluohs dna yticorta na si nerdlihc gnilaets taht eerga nac ydobyreve kniht I.
won noitca ekat ot evah ew neht ,noitan yna ot  tub
,eniarkU ni ylno ton niaga neppah reve ot siht tnaw tnod ew fI.